# 02 - 检查 Perez SLE PBMC 疾病队列

来源: GSE174188

论文涉及: 261供体、354样本

RISE预处理后参考规模: 1,263,673细胞 × 2,161基因（仅用于核对）

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('..')
data_dir = base / 'data' / 'perez_sle_pbmc'
print(f'数据目录: {data_dir.resolve()}')

# 查看样本元数据
meta_file = data_dir / 'metadata' / 'GSE174188_samples.json'
if meta_file.exists():
    with open(meta_file) as f:
        meta = json.load(f)
    samples = meta.get('samples', [])
    print(f'\n样本数: {len(samples)}')
    print(f'前3个样本:')
    for s in samples[:3]:
        print(f'  {s["gsm"]}: {s["title"]}')
        print(f'    characteristics: {s.get("characteristics", {})}')
else:
    print('样本元数据不存在，请先运行 download_perez_sle.py --metadata-only')

## 供体与样本统计

In [ ]:
if meta_file.exists():
    with open(meta_file) as f:
        meta = json.load(f)
    samples = meta.get('samples', [])

    # 收集所有characteristics字段
    all_chars = set()
    for s in samples:
        all_chars.update(s.get('characteristics', {}).keys())
    print(f'所有元数据字段: {sorted(all_chars)}')

    # 供体统计
    donor_fields = [c for c in all_chars if any(k in c.lower() for k in ['donor', 'subject', 'patient', 'individual'])]
    if donor_fields:
        donors = set()
        for s in samples:
            for df in donor_fields:
                if df in s.get('characteristics', {}):
                    donors.add(s['characteristics'][df])
        print(f'\n供体字段: {donor_fields}')
        print(f'供体数: {len(donors)}')
        print(f'论文报告: 261供体, 354样本')
        if len(donors) != 261:
            print(f'[注意] 供体数({len(donors)})与论文报告(261)不一致，需核对')
    else:
        print('\n未找到供体字段，需手动核对元数据')

    # 疾病状态
    disease_fields = [c for c in all_chars if any(k in c.lower() for k in ['disease', 'status', 'condition', 'diagnosis', 'sle'])]
    if disease_fields:
        print(f'\n疾病字段: {disease_fields}')
        for df in disease_fields:
            vals = [s['characteristics'].get(df, 'N/A') for s in samples]
            from collections import Counter
            print(f'  {df}: {dict(Counter(vals))}')
else:
    print('请先下载元数据')

## 检查表达矩阵

In [ ]:
h5ad_files = list((data_dir / 'processed').glob('*.h5ad'))
if h5ad_files:
    import anndata
    adata = anndata.read_h5ad(h5ad_files[0])
    print(f'形状: {adata.shape} (细胞 × 基因)')
    print(f'X dtype: {adata.X.dtype}')
    print(f'obs列: {list(adata.obs.columns)}')
    print(f'表达类型: {adata.uns.get("expression_type", "未标记")}')
    print(f'\n参考规模(仅核对): 1263673细胞 × 2161基因')
else:
    print('未找到h5ad表达矩阵。')
    print('参考规模(仅核对): 1263673细胞 × 2161基因')
    print('注意: 这是RISE预处理后的规模，原始文件维度可能不同。')

## 重要提醒

In [ ]:
print('=== 重要提醒 ===')
print('1. 不能将重复样本当作独立供体（261供体 vs 354样本）')
print('2. 保留供体ID、样本ID及批次信息')
print('3. RISE预处理后规模仅用于核对，不代表原始文件维度')
print('4. 不将归一化表达误存成原始计数')
print('5. 转换为AnnData时记录转换过程和表达值类型')